# Does φ Structured Sparsity Actually Reduce Inference Cost?

This notebook moves from **sparsity as a structural observation** to an explicit **computation + memory cost model**.

We compare:

1. FP32 dense reference
2. INT8 / binary digit planes
3. φ digit planes using canonical minimum-popcount representations
4. φ digit planes rewritten **exactly** to maximize block sparsity

The key question is:

> Does the extra block sparsity created by φ redundancy save enough work to compensate for the fact that φ coefficients are less hardware-friendly than powers of two?

We therefore measure:

- active digit-plane blocks,
- skipped block fraction,
- scalar contribution count,
- estimated sparse storage / memory traffic,
- sparse metadata overhead,
- relative operation cost,
- **break-even φ block cost**.

No claim of real speedup is made from Python timing. This notebook establishes the theoretical/computational conditions under which a specialized implementation could win.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math, itertools
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)


## 1. Train the same MNIST baseline


In [ ]:
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784,128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):
        x=x.view(x.size(0),-1)
        return self.fc2(self.relu(self.fc1(x)))

model=SimpleMLP()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

for epoch in range(3):
    model.train()
    total_loss=0
    for images,labels in train_loader:
        optimizer.zero_grad()
        loss=criterion(model(images),labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

def evaluate(model):
    model.eval()
    correct=total=0
    with torch.no_grad():
        for images,labels in test_loader:
            pred=model(images).argmax(1)
            correct += (pred==labels).sum().item()
            total += labels.numel()
    return correct/total

W=model.fc1.weight.detach().clone()
W_original=W.clone()
fp32_acc=evaluate(model)

print("Weight matrix:",tuple(W.shape))
print("Weights:",W.numel())
print(f"FP32 accuracy: {100*fp32_acc:.2f}%")


## 2. INT8 binary representation


In [ ]:
qmax=127
int8_scale=W.abs().max()/qmax
Q8=torch.clamp(torch.round(W/int8_scale),-qmax,qmax).to(torch.int64)
W_int8=Q8.float()*int8_scale

def binary_bits_from_codes(Q):
    mag=Q.abs().flatten().cpu().numpy().astype(np.int64)
    return np.stack([((mag>>k)&1) for k in range(7)],axis=1).astype(np.int8)

binary_bits=binary_bits_from_codes(Q8)

int8_mse=torch.mean((W-W_int8)**2).item()
with torch.no_grad():
    model.fc1.weight.copy_(W_int8)
int8_acc=evaluate(model)
with torch.no_grad():
    model.fc1.weight.copy_(W_original)

print("INT8 MSE:",int8_mse)
print(f"INT8 accuracy: {100*int8_acc:.2f}%")
print("Average active binary magnitude bits:",binary_bits.sum(1).mean())


## 3. φ codebook, redundancy, and quantization


In [ ]:
phi=(1+math.sqrt(5))/2
K=8
phi_powers=np.array([phi**(-k) for k in range(1,K+1)],dtype=np.float64)

groups=defaultdict(list)
for bits in itertools.product([0,1],repeat=K):
    value=float(np.dot(bits,phi_powers))
    groups[round(value,10)].append(tuple(bits))

phi_keys=list(groups.keys())
phi_values=np.array(phi_keys,dtype=np.float64)

canonical={
    key:min(reps,key=lambda b:(sum(b),b))
    for key,reps in groups.items()
}

phi_scale=W.abs().max().item()/phi_values.max()
flatW=W.flatten().cpu().numpy()

selected_keys=[]
W_phi_flat=np.zeros_like(flatW,dtype=np.float32)

for i,w in enumerate(flatW):
    target=abs(float(w))/phi_scale
    idx=int(np.argmin(np.abs(phi_values-target)))
    key=phi_keys[idx]
    selected_keys.append(key)
    W_phi_flat[i]=(-1 if w<0 else 1)*phi_scale*key

W_phi=torch.tensor(W_phi_flat.reshape(W.shape),dtype=W.dtype)
phi_local_bits=np.array([canonical[k] for k in selected_keys],dtype=np.int8)

phi_mse=torch.mean((W-W_phi)**2).item()
with torch.no_grad():
    model.fc1.weight.copy_(W_phi)
phi_acc=evaluate(model)
with torch.no_grad():
    model.fc1.weight.copy_(W_original)

print("Unique positive φ values:",len(groups))
print("φ MSE:",phi_mse)
print(f"φ accuracy: {100*phi_acc:.2f}%")
print("Canonical φ avg active digits:",phi_local_bits.sum(1).mean())


## 4. Exact φ block rewriting

For each spatial block, we choose equivalent φ representations while preserving **every quantized numerical value exactly**.

The objective strongly minimizes the number of active digit planes inside each spatial block, then minimizes total active digits.

Thus any accuracy/MSE difference between canonical φ and block-optimized φ is exactly zero apart from floating-point reconstruction tolerance.


In [ ]:
def block_slices(shape,br,bc):
    R,C=shape
    for r0 in range(0,R,br):
        for c0 in range(0,C,bc):
            yield slice(r0,min(r0+br,R)),slice(c0,min(c0+bc,C))

def exact_optimize_phi_blocks(selected_keys,shape,block_size,block_penalty=1_000_000):
    R,C=shape
    key_matrix=np.array(selected_keys,dtype=object).reshape(R,C)
    result=np.zeros((R,C,K),dtype=np.int8)
    cache={}

    def best_rep(key,mask):
        ck=(key,mask)
        if ck in cache:
            return cache[ck]
        candidates=[]
        for b in groups[key]:
            valid=True
            for k,bit in enumerate(b):
                if bit and not ((mask>>k)&1):
                    valid=False
                    break
            if valid:
                candidates.append(b)
        ans=None if not candidates else min(candidates,key=lambda b:(sum(b),b))
        cache[ck]=ans
        return ans

    for rs,cs in block_slices(shape,block_size,block_size):
        kb=key_matrix[rs,cs]
        unique=set(kb.flatten())
        best_cost=float("inf")
        best_map=None

        for mask in range(1,2**K):
            reps={}
            feasible=True
            for key in unique:
                rep=best_rep(key,mask)
                if rep is None:
                    feasible=False
                    break
                reps[key]=rep
            if not feasible:
                continue

            tmp=np.array([reps[k] for k in kb.flatten()],dtype=np.int8)
            active_planes=np.count_nonzero(tmp.sum(0))
            digits=int(tmp.sum())
            cost=block_penalty*active_planes+digits

            if cost<best_cost:
                best_cost=cost
                best_map=reps

        for i in range(kb.shape[0]):
            for j in range(kb.shape[1]):
                result[rs.start+i,cs.start+j]=best_map[kb[i,j]]

    flat=result.reshape(-1,K)
    decoded=flat.astype(np.float64)@phi_powers
    assert np.allclose(decoded,np.array(selected_keys),atol=1e-8)
    return flat


In [ ]:
block_sizes=[4,8,16,32]
phi_exact={}

for bs in block_sizes:
    print("Optimizing",bs,"x",bs)
    phi_exact[bs]=exact_optimize_phi_blocks(selected_keys,W.shape,bs)


# 5. Cost-model definitions

A **digit-plane block** is one spatial block in one digit plane.

For example, with 4×4 spatial blocks:

- binary has 7 magnitude planes,
- φ has 8 magnitude planes.

If a block in a digit plane is all zero, a block-sparse implementation can potentially skip it.

We calculate several quantities:

### Active block count
Number of digit-plane blocks containing at least one `1`.

### Dense scalar slots avoided
Each full \(b\times b\) block contains \(b^2\) possible weight contributions. If a block is skipped, those positions need not be visited by a block-level kernel.

### Active digits
Actual number of `1` digits. This measures fine-grained arithmetic activity.

### Sparse storage model
For each active block we store:

- the block's binary mask: \(b^2\) bits,
- a block index: configurable metadata bits.

This is intentionally simple and transparent.

We also separately include one sign bit per nonzero quantized weight. A real implementation could compress signs differently.


In [ ]:
def plane_array(bits,shape):
    R,C=shape
    P=bits.shape[1]
    return np.stack([bits[:,p].reshape(R,C) for p in range(P)],axis=0)

def cost_stats(bits,shape,block_size,metadata_bits=16,sign_bits_per_weight=1):
    planes=plane_array(bits,shape)
    P,R,C=planes.shape

    total_blocks=0
    active_blocks=0
    actual_block_cells=0

    for p in range(P):
        for rs,cs in block_slices(shape,block_size,block_size):
            total_blocks += 1
            block=planes[p,rs,cs]
            if np.any(block):
                active_blocks += 1
                actual_block_cells += block.size

    zero_blocks=total_blocks-active_blocks
    active_digits=int(bits.sum())

    # Payload: one literal bit per digit position inside every stored active block.
    payload_bits=actual_block_cells
    metadata_total_bits=active_blocks*metadata_bits

    # Conservative simple sign model.
    sign_bits=shape[0]*shape[1]*sign_bits_per_weight

    total_sparse_bits=payload_bits+metadata_total_bits+sign_bits

    return {
        "planes":P,
        "total_blocks":total_blocks,
        "active_blocks":active_blocks,
        "zero_blocks":zero_blocks,
        "zero_fraction":zero_blocks/total_blocks,
        "active_digits":active_digits,
        "avg_active_digits":active_digits/(R*C),
        "block_payload_bits":payload_bits,
        "metadata_bits":metadata_total_bits,
        "sign_bits":sign_bits,
        "sparse_storage_bits":total_sparse_bits,
        "sparse_storage_bytes":total_sparse_bits/8,
        "visited_block_cells":actual_block_cells
    }


## 6. Compare canonical binary, canonical φ, and exact block-optimized φ


In [ ]:
all_stats=[]

for bs in block_sizes:
    methods=[
        ("BINARY_INT8",binary_bits),
        ("PHI_CANONICAL",phi_local_bits),
        ("PHI_BLOCK_OPT",phi_exact[bs])
    ]

    for name,bits in methods:
        s=cost_stats(bits,W.shape,bs,metadata_bits=16)
        s["method"]=name
        s["block_size"]=bs
        all_stats.append(s)

print(
    f"{'Method':<16}{'Block':>7}{'Zero%':>10}{'Active blocks':>15}"
    f"{'Active digits':>16}{'Visited cells':>16}{'Sparse KB':>12}"
)

for s in all_stats:
    print(
        f"{s['method']:<16}{s['block_size']:>7}"
        f"{100*s['zero_fraction']:>9.2f}%"
        f"{s['active_blocks']:>15}"
        f"{s['active_digits']:>16}"
        f"{s['visited_block_cells']:>16}"
        f"{s['sparse_storage_bytes']/1024:>12.2f}"
    )


# 7. Break-even computation cost

Suppose one active binary digit-plane block costs \(C_B\).

Suppose one active φ digit-plane block costs \(C_\phi\).

Ignoring fixed overhead for the moment:

\[
\text{binary cost}=N_B C_B
\]

\[
\text{φ cost}=N_\phi C_\phi.
\]

φ wins when

\[
N_\phi C_\phi < N_B C_B.
\]

Therefore:

\[
\frac{C_\phi}{C_B}<\frac{N_B}{N_\phi}.
\]

The right-hand side is the **break-even φ block-cost multiplier**.

For example, a value of `1.20×` means a φ block could be 20% more expensive than a binary block and still break even in this simplified model.


In [ ]:
print(f"{'Block':>8}{'Binary blocks':>16}{'φ blocks':>12}{'φ/Binary blocks':>18}{'Break-even Cφ/CB':>20}")

break_even=[]

for bs in block_sizes:
    b=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="BINARY_INT8")
    p=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="PHI_BLOCK_OPT")

    ratio=p["active_blocks"]/b["active_blocks"]
    be=b["active_blocks"]/p["active_blocks"]

    break_even.append((bs,be))

    print(
        f"{str(bs)+'x'+str(bs):>8}"
        f"{b['active_blocks']:>16}"
        f"{p['active_blocks']:>12}"
        f"{ratio:>18.4f}"
        f"{be:>20.4f}x"
    )


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(
    [x[0] for x in break_even],
    [x[1] for x in break_even],
    marker="o"
)
plt.axhline(1.0,linestyle="--")
plt.xlabel("Block side length")
plt.ylabel("Maximum φ block cost / binary block cost")
plt.title("φ Break-even Arithmetic Cost")
plt.grid(alpha=0.25)
plt.show()


# 8. More realistic mixed cost: block overhead + active digits

Counting only blocks may over-reward structured sparsity because our φ block optimization sometimes increases the number of `1` digits.

Use:

\[
C = c_{\text{block}}N_{\text{blocks}} + c_{\text{digit}}N_{\text{digits}}.
\]

`c_block` models setup / loading / indexing / block scheduling.

`c_digit` models fine-grained arithmetic contributions.

We sweep their relative importance.


In [ ]:
block_overheads=[0,1,2,4,8,16,32,64]

for bs in block_sizes:
    b=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="BINARY_INT8")
    p=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="PHI_BLOCK_OPT")

    binary_cost=[]
    phi_cost=[]

    for cb in block_overheads:
        # Normalize one active digit contribution to cost 1.
        binary_cost.append(cb*b["active_blocks"] + b["active_digits"])
        phi_cost.append(cb*p["active_blocks"] + p["active_digits"])

    plt.figure(figsize=(8,5))
    plt.plot(block_overheads,binary_cost,marker="o",label="Binary")
    plt.plot(block_overheads,phi_cost,marker="o",label="φ block-optimized")
    plt.xlabel("Cost assigned to each active block")
    plt.ylabel("Relative modeled cost")
    plt.title(f"{bs}×{bs}: Mixed Block + Digit Cost")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


## 9. Solve the block-overhead threshold analytically

Assume an active digit costs 1 unit in both representations.

We solve for block overhead \(h\):

\[
hN_\phi + D_\phi < hN_B + D_B.
\]

Thus, when \(N_\phi<N_B\),

\[
h > \frac{D_\phi-D_B}{N_B-N_\phi}.
\]

This tells us how expensive block-level overhead must be before φ's additional structured sparsity compensates for its active-digit count.


In [ ]:
print(f"{'Block':>8}{'Δ digits φ-bin':>18}{'Blocks saved':>15}{'Break-even overhead':>22}")

for bs in block_sizes:
    b=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="BINARY_INT8")
    p=next(s for s in all_stats if s["block_size"]==bs and s["method"]=="PHI_BLOCK_OPT")

    delta_digits=p["active_digits"]-b["active_digits"]
    blocks_saved=b["active_blocks"]-p["active_blocks"]

    if blocks_saved>0:
        h=delta_digits/blocks_saved
        interpretation = h
    else:
        interpretation = float("inf")

    print(
        f"{str(bs)+'x'+str(bs):>8}"
        f"{delta_digits:>18}"
        f"{blocks_saved:>15}"
        f"{interpretation:>22.4f}"
    )


# 10. Memory traffic / storage sensitivity

Sparse block formats have metadata overhead. We therefore sweep the assumed metadata cost per active block.

This is not a claim about a particular sparse format. It tells us how sensitive the apparent memory advantage is to indexing overhead.

For reference, dense storage is approximately:

- INT8: 8 bits/weight
- literal φ: 8 magnitude digits + sign = 9 bits/weight

A compact φ codebook index could have a different storage cost, but then direct digit-plane execution would require decoding.


In [ ]:
metadata_options=[0,4,8,16,32,64]
N=W.numel()

dense_int8_bits=8*N
dense_phi_literal_bits=9*N

print("Dense INT8 KB:",dense_int8_bits/8/1024)
print("Dense literal φ KB:",dense_phi_literal_bits/8/1024)

for bs in block_sizes:
    bin_mem=[]
    phi_mem=[]

    for mb in metadata_options:
        b=cost_stats(binary_bits,W.shape,bs,metadata_bits=mb)
        p=cost_stats(phi_exact[bs],W.shape,bs,metadata_bits=mb)
        bin_mem.append(b["sparse_storage_bytes"]/1024)
        phi_mem.append(p["sparse_storage_bytes"]/1024)

    plt.figure(figsize=(8,5))
    plt.plot(metadata_options,bin_mem,marker="o",label="Binary sparse")
    plt.plot(metadata_options,phi_mem,marker="o",label="φ sparse")
    plt.axhline(dense_int8_bits/8/1024,linestyle="--",label="Dense INT8")
    plt.axhline(dense_phi_literal_bits/8/1024,linestyle=":",label="Dense literal φ")
    plt.xlabel("Metadata bits per active block")
    plt.ylabel("Estimated storage / bytes read (KB)")
    plt.title(f"{bs}×{bs}: Sparse Memory Sensitivity")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


# 11. Memory break-even summary at 16 metadata bits/block


In [ ]:
print(f"{'Method':<16}{'Block':>8}{'Estimated KB':>16}{'vs dense INT8':>18}")

for bs in block_sizes:
    for method,bits in [
        ("BINARY",binary_bits),
        ("PHI_OPT",phi_exact[bs])
    ]:
        s=cost_stats(bits,W.shape,bs,metadata_bits=16)
        kb=s["sparse_storage_bytes"]/1024
        ratio=s["sparse_storage_bits"]/dense_int8_bits
        print(f"{method:<16}{str(bs)+'x'+str(bs):>8}{kb:>16.2f}{ratio:>17.3f}x")


# 12. Optional Python prototype timing

**Interpret carefully.**

This section executes only active blocks in Python/NumPy. It demonstrates that the representations can be traversed as block-sparse structures, but it is **not a hardware benchmark**.

Python loop overhead can dominate the actual arithmetic. Do not use these timings as evidence that φ is faster or slower in optimized inference.


In [ ]:
import time

def prepare_active_blocks(bits,shape,block_size):
    planes=plane_array(bits,shape)
    active=[]

    for p in range(planes.shape[0]):
        for rs,cs in block_slices(shape,block_size,block_size):
            block=planes[p,rs,cs]
            if np.any(block):
                active.append((p,rs,cs,block.astype(np.float32)))

    return active

def sparse_plane_matvec(active_blocks,coeffs,scale,x,rows):
    y=np.zeros(rows,dtype=np.float32)

    for p,rs,cs,block in active_blocks:
        y[rs] += scale*coeffs[p]*(block @ x[cs])

    return y

# Signs are not incorporated into this simple timing prototype.
# The timing is only for block traversal + digit-plane contribution mechanics.

bs=4
bin_active=prepare_active_blocks(binary_bits,W.shape,bs)
phi_active=prepare_active_blocks(phi_exact[bs],W.shape,bs)

bin_coeffs=np.array([2**k for k in range(7)],dtype=np.float32)
phi_coeffs=phi_powers.astype(np.float32)

x=np.random.randn(W.shape[1]).astype(np.float32)

def benchmark(fn,repeats=100):
    for _ in range(10):
        fn()
    times=[]
    for _ in range(repeats):
        t0=time.perf_counter()
        fn()
        times.append(time.perf_counter()-t0)
    return np.median(times),np.mean(times)

tb=benchmark(lambda:sparse_plane_matvec(bin_active,bin_coeffs,float(int8_scale),x,W.shape[0]))
tp=benchmark(lambda:sparse_plane_matvec(phi_active,phi_coeffs,float(phi_scale),x,W.shape[0]))

print("WARNING: Python prototype only; not a hardware speed comparison.")
print(f"Binary median: {tb[0]*1e3:.3f} ms")
print(f"φ median:      {tp[0]*1e3:.3f} ms")
print(f"φ / binary timing ratio: {tp[0]/tb[0]:.3f}x")


# 13. Final decision table

Interpretation:

- **Active-block ratio < 1:** φ visits fewer digit-plane blocks.
- **Break-even multiplier > 1:** φ blocks may be somewhat more expensive while still winning under a pure block-count model.
- **Mixed-cost threshold:** tells us how important block overhead must be before φ's block structure offsets its digit count.
- **Sparse-memory ratio < 1 relative to binary:** suggests a possible memory-traffic/storage advantage under the stated sparse format assumptions.
- Python timing is diagnostic only.

The next step after a promising cost model is an optimized C++/SIMD or CUDA kernel with sign handling and actual hardware counters.


In [ ]:
print("="*76)
print("FINAL COMPUTATIONAL SUMMARY")
print("="*76)

for bs in block_sizes:
    b=cost_stats(binary_bits,W.shape,bs,metadata_bits=16)
    p=cost_stats(phi_exact[bs],W.shape,bs,metadata_bits=16)

    block_ratio=p["active_blocks"]/b["active_blocks"]
    be=b["active_blocks"]/p["active_blocks"]
    blocks_saved=b["active_blocks"]-p["active_blocks"]
    delta_digits=p["active_digits"]-b["active_digits"]

    if blocks_saved>0:
        overhead_threshold=delta_digits/blocks_saved
    else:
        overhead_threshold=float("inf")

    print(f"\n{bs}x{bs}")
    print(f"  Binary zero blocks: {100*b['zero_fraction']:.2f}%")
    print(f"  φ zero blocks:      {100*p['zero_fraction']:.2f}%")
    print(f"  Active-block ratio φ/binary: {block_ratio:.4f}")
    print(f"  Pure block-cost break-even Cφ/CB: {be:.4f}x")
    print(f"  Binary active digits: {b['active_digits']}")
    print(f"  φ active digits:      {p['active_digits']}")
    print(f"  Blocks saved by φ:    {blocks_saved}")
    print(f"  Δ active digits:      {delta_digits}")
    print(f"  Mixed-model block-overhead threshold: {overhead_threshold:.4f}")
    print(f"  Binary sparse KB (16-bit metadata): {b['sparse_storage_bytes']/1024:.2f}")
    print(f"  φ sparse KB (16-bit metadata):      {p['sparse_storage_bytes']/1024:.2f}")

print("\nAccuracy is unchanged by exact φ rewrites:")
print(f"  FP32: {100*fp32_acc:.2f}%")
print(f"  INT8: {100*int8_acc:.2f}%")
print(f"  φ:    {100*phi_acc:.2f}%")
